# 04 Gold - Retail

**Audience:** participants learning the AIDP medallion pattern with PySpark.

**Prerequisites:** use the lab's shared compute and run the previous notebook first.

**Learning goal:** Builds the industry KPIs from accepted Silver records.

## Outline

1. Inspect the participant-scoped inputs.
2. Transform and persist this medallion layer.
3. Register external tables when this layer owns them.
4. Verify the row counts printed by the final statements.


In [ ]:
import re
# oidlUtils is injected by AIDP Workbench; no import is required.

def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name, "")
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")
catalog_name = required_parameter("catalog_name")

participant_match = re.fullmatch(r"u([1-9][0-9]*)", participant_key)
if participant_match is None or int(participant_match.group(1)) < 101:
    raise ValueError("Invalid participant_key")
if lab_id != 'retail':
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")
if catalog_name != f"{participant_key}_aidp_lab":
    raise ValueError("Invalid participant catalog")
spark.conf.set("spark.aidp.lineage.enabled", "true")

def table(layer, logical_name):
    prefix = f"{lab_id}_"
    physical_name = logical_name if logical_name.startswith(prefix) else prefix + logical_name
    return f"{catalog_name}.oci_{layer}.{participant_key}_{physical_name}"

from pyspark.sql import functions as F

silver = {"customers": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/retail/customers/", "order_items": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/retail/order_items/", "orders": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/retail/orders/", "products": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/retail/products/"}
gold = {"customer_value": f"oci://{bucket_name}@{objectstorage_namespace}/04_gold/users/{participant_key}/retail/retail_customer_value/", "product_daily": f"oci://{bucket_name}@{objectstorage_namespace}/04_gold/users/{participant_key}/retail/retail_product_daily/"}
customers = spark.table(table("silver", "customers"))
products = spark.table(table("silver", "products"))
orders = spark.table(table("silver", "orders"))
items = spark.table(table("silver", "order_items"))
lines = (items
    .join(
        orders.select("participant_key", "order_id", "customer_id", "order_time", "order_status"),
        ["participant_key", "order_id"],
    )
    .join(products.select("participant_key", "product_id", "unit_cost"), ["participant_key", "product_id"])
    .withColumn("gross", F.col("quantity") * F.col("unit_price"))
    .withColumn("net", F.col("gross") - F.col("discount_amount")))
customer_value = (lines.groupBy("participant_key", "customer_id")
    .agg(F.countDistinct("order_id").alias("order_count"), F.sum("quantity").alias("units"), F.sum("gross").alias("gross_revenue"), F.sum("discount_amount").alias("discount_amount"), F.sum("net").alias("net_revenue"), F.max("order_time").alias("last_order_at"))
    .withColumn("average_order_value", F.round(F.col("net_revenue") / F.col("order_count"), 2))
    .select("participant_key", "customer_id", "order_count", "units", "gross_revenue", "discount_amount", "net_revenue", "average_order_value", "last_order_at"))
customer_value.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table("gold", "customer_value"))
product_daily = (lines.withColumn("order_date", F.to_date("order_time"))
    .groupBy("participant_key", "order_date", "product_id")
    .agg(F.sum("quantity").alias("units"), F.sum("net").alias("net_revenue"), F.sum(F.col("net") - F.col("quantity") * F.col("unit_cost")).alias("gross_margin"), F.sum(F.when(F.col("order_status") == "refunded", F.col("quantity")).otherwise(0)).alias("refunded_units")))
product_daily.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table("gold", "product_daily"))
customer_value.show(20, truncate=False)

for table_name in gold:
    row_count = spark.table(table("gold", table_name)).count()
    assert row_count > 0, f"Gold table {table_name} is empty"
    print(f"Gold {table_name}: {row_count} rows")


## Expected result

Two non-empty, industry-specific aggregate Delta tables are registered.

**Exercise:** rerun this notebook and confirm that counts do not increase. All writes use
participant-exclusive paths and overwrite mode, so a second run is idempotent.

**Common pitfall:** do not replace the participant paths with shared locations. That would mix
different students' data. As an extension, query the registered tables with `spark.sql`.
